In [55]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

RAW_DATA = PROJECT_ROOT / "FIFA WC Dataset"
CLEANED_DATA = PROJECT_ROOT / "cleaned_data"
PROCESSED_DATA = PROJECT_ROOT / "processed_data"

team_matches = pd.read_csv(PROCESSED_DATA / "team_matches.csv")
saved_stats = pd.read_csv(PROCESSED_DATA / "team_stats.csv")
results = pd.read_csv(CLEANED_DATA / "results_2000_onwards.csv")

In [56]:
results["date"] = pd.to_datetime(results["date"], dayfirst = True)

In [57]:
# result_played will only store the matches which are already played. Future matches are removed (until they happen).
results_played = results[results["home_score"].notna() & results["away_score"].notna()].copy()

In [58]:
results_played = results_played.sort_values("date").reset_index(drop=True)

In [59]:
INITIAL_ELO = 1500
HOME_ADVANTAGE = 100

def get_k_value(tournament):

    if tournament == "Friendly":
        return 10

    elif tournament == "FIFA World Cup":
        return 60

    elif tournament == "FIFA World Cup qualification":
        return 40

    elif tournament in [
        "UEFA Euro",
        "Copa América",
        "AFC Asian Cup",
        "African Cup of Nations",
        "CONCACAF Gold Cup",
        "OFC Nations Cup"
    ]:
        return 35

    elif "qualification" in tournament.lower():
        return 25

    elif "nations league" in tournament.lower():
        return 25

    else:
        return 20
    
elo_ratings = {}

# Created a get_rating function which will get the elo ratings of a team 

def get_rating(team):
    if team not in elo_ratings:
        elo_ratings[team] = INITIAL_ELO
    return elo_ratings[team]

# Created a new get_result function which will store the results Win, Draw, Lose, i.e. W, D, L based on the home-away scores.

def get_result(goals_for, goals_against):
    if goals_for > goals_against:
        return "W"
    elif goals_for < goals_against:
        return "L"
    else:
        return "D"
    
# store match-by-match elo history 
history_records = []

In [60]:
first_match = results_played.iloc[0]

first_match

date          2000-01-04 00:00:00
home_team                   Egypt
away_team                    Togo
home_score                      2
away_score                      1
tournament               Friendly
city                        Aswan
country                     Egypt
neutral                     False
Name: 0, dtype: object

In [61]:
home_team = first_match["home_team"]
away_team = first_match["away_team"]

home_score = first_match["home_score"]
away_score = first_match["away_score"]

print(home_team, home_score)
print(away_team, away_score)

Egypt 2
Togo 1


In [62]:
home_rating = get_rating(home_team)
away_rating = get_rating(away_team)

print(home_rating)
print(away_rating)


1500
1500


In [63]:
# Calculation if teams draw a match.

expected_home = 1 / (
    1 + 10 ** ((away_rating - home_rating) / 400)
)

expected_away = 1 - expected_home

print(expected_home)
print(expected_away)

0.5
0.5


In [64]:
# Giving points based on match results. if a team Wins it gets 1 point, a Lose gets 0 and 0.5 for a Draw.

if home_score > away_score:
    actual_home = 1
    actual_away = 0

elif home_score < away_score:
    actual_home = 0
    actual_away = 1

else:
    actual_home = 0.5
    actual_away = 0.5

print(actual_home)
print(actual_away)

1
0


In [65]:
def goal_difference_multiplier(goal_difference):
    """
    Returns the Elo multiplier based on goal difference.
    """

    if goal_difference == 1:
        return 1.00
    elif goal_difference == 2:
        return 1.50
    elif goal_difference == 3:
        return 1.75
    elif goal_difference == 4:
        return 2.00
    else:
        return 2.25

In [66]:
# Processing matches in chronological order.

for _, match in results_played.iterrows():

    # Match details
    date = match["date"]
    home_team = match["home_team"]
    away_team = match["away_team"]
    tournament = match["tournament"]
    neutral = match["neutral"]
    # Get K value for this tournament
    k = get_k_value(tournament)


    home_score = match["home_score"]
    away_score = match["away_score"]

        # Goal difference multiplier
    goal_difference = abs(home_score - away_score)
    gd_multiplier = goal_difference_multiplier(goal_difference)

    # Current ratings
    home_rating = get_rating(home_team)
    away_rating = get_rating(away_team)

    # Apply home advantage only for non-neutral matches
    effective_home_rating = home_rating
    effective_away_rating = away_rating

    if not neutral:
        effective_home_rating += HOME_ADVANTAGE

    # Expected scores
    expected_home = 1 / (
        1 + 10 ** (
            (effective_away_rating - effective_home_rating) / 400
        )
    )
    expected_away = 1 - expected_home

    # Actual scores
    if home_score > away_score:
        actual_home = 1
        actual_away = 0

    elif home_score < away_score:
        actual_home = 0
        actual_away = 1

    else:
        actual_home = 0.5
        actual_away = 0.5

    # New ratings
    new_home_rating = (
        home_rating +
        k * gd_multiplier * (actual_home - expected_home)
    )

    new_away_rating = (
        away_rating +
        k * gd_multiplier * (actual_away - expected_away)
    )

    # Results
    home_result = get_result(home_score, away_score)
    away_result = get_result(away_score, home_score)

    # Home team history row
    history_records.append({
        "date": date,
        "team": home_team,
        "opponent": away_team,
        "tournament": tournament,
        "goals_for": home_score,
        "goals_against": away_score,
        "result": home_result,
        "elo_before": home_rating,
        "elo_after": new_home_rating
    })

    # Away team history row
    history_records.append({
        "date": date,
        "team": away_team,
        "opponent": home_team,
        "tournament": tournament,
        "goals_for": away_score,
        "goals_against": home_score,
        "result": away_result,
        "elo_before": away_rating,
        "elo_after": new_away_rating
    })

    # Update ratings dictionary
    elo_ratings[home_team] = new_home_rating
    elo_ratings[away_team] = new_away_rating


In [67]:
elo_history_v3 = pd.DataFrame(history_records)

In [68]:
elo_ratings_v3 = pd.DataFrame(
    elo_ratings.items(),
    columns=["team", "elo_rating"]
)

elo_ratings_v3 = elo_ratings_v3.sort_values(
    by="elo_rating",
    ascending=False
)
elo_ratings_v3 = elo_ratings_v3.sort_values(
    by="elo_rating",
    ascending=False
).reset_index(drop=True)

In [69]:
elo_ratings_v3.head(10)

,team,elo_rating
0,Argentina,2063.031955
1,France,2035.578715
2,Spain,1994.092512
3,Brazil,1985.056909
4,England,1953.566348
5,Colombia,1952.931482
6,Netherlands,1927.259187
7,Morocco,1924.293672
8,Portugal,1921.895343
9,Mexico,1895.998555


In [70]:
# Round history
elo_history_v3["elo_before"] = elo_history_v3["elo_before"].round(2)
elo_history_v3["elo_after"] = elo_history_v3["elo_after"].round(2)

# Round final ratings
elo_ratings_v3["elo_rating"] = elo_ratings_v3["elo_rating"].round(2)

# Add ranking
elo_ratings_v3.insert(
    0,
    "rank",
    range(1, len(elo_ratings_v3) + 1)
)

In [71]:
elo_ratings_v3.to_csv(PROCESSED_DATA / "elo_ratings_v3.csv", index = False)
elo_history_v3.to_csv(PROCESSED_DATA / "elo_history_v3.csv", index = False)

## Elo Rating V3

This version extends the V2.5 Elo system by introducing **home advantage** while keeping all previous improvements unchanged.

### Features
- Tournament-weighted K values
- Goal difference multiplier
- Home advantage (+100 Elo for non-neutral matches)
- Chronological Elo rating updates
- Complete Elo rating history for every team

### Outputs
- `elo_ratings_v3.csv`
- `elo_history_v3.csv`

### Validation
- Mean Elo remains **1500**, preserving the zero-sum property.
- Ratings were verified against V2.5 and showed realistic changes without abnormal inflation or deflation.